# Fair Code — Audit 06: Healthcare Readmission — Clinical Bias

> *A hospital readmission model flags patients for high clinical risk using payer code and discharge destination — variables that measure insurance access, not medical severity.*

**Dataset:** Diabetes 130-US Hospitals 1999–2008 — `diabetic_data.csv` (101,766 records)  
**Protected attributes:** Race, Gender, Age  
**Proxy variables:** `payer_code` (Medicaid rate differs by race), `discharge_disposition_id` (SNF access differs by race), `medical_specialty` (encodes insurance access), `number_inpatient` (prior hospitalisations encode preventive care gaps)  
**Fairness metric:** Demographic Parity  
**Model:** Random Forest Classifier  

---

Pipeline:
1. Load and explore the dataset
2. Identify the proxy variables (four of them)
3. Train the biased model (protected attributes + proxies included)
4. Measure the fairness gaps across three dimensions
5. Train the fair model (all removed)
6. Compare results

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency

# Consistent styling across all Fair Code notebooks
plt.rcParams.update({
    'figure.facecolor': '#0d0f14',
    'axes.facecolor': '#131620',
    'axes.edgecolor': '#1e2130',
    'axes.labelcolor': '#b0aec0',
    'xtick.color': '#b0aec0',
    'ytick.color': '#b0aec0',
    'text.color': '#d4cfc0',
    'grid.color': '#1e2130',
    'grid.linestyle': '--',
    'font.family': 'monospace',
    'figure.dpi': 120
})

ACCENT = '#c9a84c'   # Fair Code gold
DANGER = '#9b2335'   # red — bias
SAFE   = '#4a7c6f'   # teal — mitigated
MUTED  = '#b0aec0'

print('Libraries loaded.')

## 2. Load and Explore the Dataset

In [ ]:
df_raw = pd.read_csv('Healthcare Readmission/diabetic_data.csv')
print(f'Dataset: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')
print(f'\nColumns: {list(df_raw.columns)}')
df_raw.head(3)

In [ ]:
df = df_raw.copy()

# Remove invalid entries
df = df[~df['race'].isin(['?'])]
df = df[df['gender'] != 'Unknown/Invalid']

# Target: 1 = readmitted within 30 days (flagged as high clinical risk)
#         0 = not readmitted within 30 days
# In a real hospital tool this flag drives discharge planning and resource allocation
df['target'] = (df['readmitted'] == '<30').astype(int)

# Protected attribute flags — retained for fairness measurement only
df['is_female']   = (df['gender'] == 'Female').astype(int)
df['is_minority'] = (~df['race'].isin(['Caucasian', 'Asian'])).astype(int)
df['age_numeric'] = df['age'].str.extract(r'\[(\d+)').astype(int)
df['is_elderly']  = (df['age_numeric'] >= 70).astype(int)

print('Protected group breakdown:')
print(f'  Female patients   : {df["is_female"].mean():.1%}')
print(f'  Racial minorities : {df["is_minority"].mean():.1%}')
print(f'  Elderly (70+)     : {df["is_elderly"].mean():.1%}')

print(f'\nOverall 30-day readmission rate: {df["target"].mean():.1%}')

print('\n30-day readmission rate by gender (raw data):')
gender_raw = df.groupby('gender')['target'].mean() * 100
for g, v in gender_raw.items():
    print(f'  {g:<10}: {v:.2f}%')
print(f'  Raw gap: {abs(gender_raw["Male"] - gender_raw["Female"]):.2f} percentage points')

print('\n30-day readmission rate by race (raw data):')
race_raw = df.groupby('is_minority')['target'].mean() * 100
print(f'  Caucasian/Asian : {race_raw[0]:.2f}%')
print(f'  Other minorities: {race_raw[1]:.2f}%')
print(f'  Raw gap: {abs(race_raw[0] - race_raw[1]):.2f} percentage points')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

# Readmission rate by gender
gender_plot = df.groupby('gender')['target'].mean() * 100
labels_g = ['Male', 'Female']
vals_g = [gender_plot['Male'], gender_plot['Female']]
bars = axes[0].bar(labels_g, vals_g, color=[MUTED, DANGER], width=0.4)
for bar, val in zip(bars, vals_g):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{val:.2f}%', ha='center', color=ACCENT, fontsize=10)
axes[0].set_title('30-day readmission by gender (raw)', color=MUTED, fontsize=10)
axes[0].set_ylabel('% readmitted within 30 days')
axes[0].set_ylim(0, 1.5)
axes[0].grid(axis='y', alpha=0.3)

# Readmission rate by race
race_plot = df.groupby('is_minority')['target'].mean() * 100
labels_r = ['Caucasian/Asian', 'Other minorities']
vals_r = [race_plot[0], race_plot[1]]
bars2 = axes[1].bar(labels_r, vals_r, color=[MUTED, DANGER], width=0.4)
for bar, val in zip(bars2, vals_r):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{val:.2f}%', ha='center', color=ACCENT, fontsize=10)
axes[1].set_title('30-day readmission by race (raw)', color=MUTED, fontsize=10)
axes[1].set_ylabel('% readmitted within 30 days')
axes[1].set_ylim(0, 1.5)
axes[1].grid(axis='y', alpha=0.3)

# Readmission rate by age group
age_plot = df.groupby('is_elderly')['target'].mean() * 100
labels_a = ['Under 70', '70+ (elderly)']
vals_a = [age_plot[0], age_plot[1]]
bars3 = axes[2].bar(labels_a, vals_a, color=[MUTED, DANGER], width=0.4)
for bar, val in zip(bars3, vals_a):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{val:.2f}%', ha='center', color=ACCENT, fontsize=10)
axes[2].set_title('30-day readmission by age (raw)', color=MUTED, fontsize=10)
axes[2].set_ylabel('% readmitted within 30 days')
axes[2].set_ylim(0, 1.5)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
print('Note: the raw gaps exist in the data before any model is trained.')

## 3. Identify the Proxy Variables

A proxy variable correlates with a protected attribute strongly enough to smuggle the bias back through the model — even after the protected column is removed.

This audit has **four** proxy variables:

| Proxy | Protected attribute | Mechanism |
|---|---|---|
| `payer_code` | Race | Medicaid rates: Hispanic 9.0%, AfricanAmerican 5.5%, Caucasian 2.7%. Insurance type encodes income, which is racially stratified. |
| `discharge_disposition_id` | Race | SNF discharge rates: Caucasian 17.3% vs AfricanAmerican 10.7%. Post-acute care access is determined by insurance and geography, not clinical need. |
| `medical_specialty` | Race, Income | Specialty access is stratified by insurance type and zip code. |
| `number_inpatient` | Race | AfricanAmerican patients average 0.70 prior inpatient visits vs 0.48 for Asian patients — a gap driven by preventive care access, not clinical severity alone. |

In [ ]:
def check_proxy(df, feature, protected_col):
    """Chi-squared test of independence. p < 0.05 = likely proxy."""
    contingency = pd.crosstab(df[feature], df[protected_col])
    chi2, p, dof, _ = chi2_contingency(contingency)
    return {'feature': feature, 'protected_attr': protected_col,
            'p_value': round(p, 6), 'is_proxy': p < 0.05}

# Test categorical proxies against race
for feat in ['payer_code', 'medical_specialty']:
    r = check_proxy(df, feat, 'race')
    print(f"{r['feature']:<25} p={r['p_value']:<12} proxy={r['is_proxy']}")

# Test discharge_disposition_id against race
r = check_proxy(df, 'discharge_disposition_id', 'race')
print(f"{r['feature']:<25} p={r['p_value']:<12} proxy={r['is_proxy']}")

# number_inpatient — continuous, use correlation
inpt_corr = df[['number_inpatient', 'is_minority']].corr().iloc[0, 1]
print(f'number_inpatient          Pearson r={inpt_corr:.4f} (positive = minorities have higher prior counts)')

In [ ]:
# payer_code x race — Medicaid rate
print('Medicaid payer rate by race:')
df['is_medicaid'] = (df['payer_code'] == 'MD').astype(int)
medicaid_race = df.groupby('race')['is_medicaid'].mean().round(3)
for r, v in medicaid_race.items():
    print(f'  {r:<22} {v:.1%}')
print('  → Medicaid encodes low income, which is racially stratified.')
print('    Hispanic: 9.0%, AfricanAmerican: 5.5%, Caucasian: 2.7%')

# discharge_disposition_id x race — SNF access
print('\nSNF (skilled nursing) discharge rate by race:')
df['discharged_to_snf'] = df['discharge_disposition_id'].isin([2, 3]).astype(int)
snf_race = df.groupby('race')['discharged_to_snf'].mean().round(3)
for r, v in snf_race.items():
    print(f'  {r:<22} {v:.1%}')
print('  → SNF access requires insurance and nearby facility.')
print('    Caucasian: 17.3% vs AfricanAmerican: 10.7%.')
print('    Lower SNF access → higher home readmission risk — encoding')
print('    structural inequality as individual clinical risk.')

# number_inpatient x race — prior hospitalisation gap
print('\nMean prior inpatient visits by race:')
prior_in = df.groupby('race')['number_inpatient'].mean().round(3)
for r, v in prior_in.items():
    print(f'  {r:<22} {v:.3f}')
print('  → AfricanAmerican patients average 0.70 prior visits vs')
print('    0.48 for Asian patients. Gap reflects differential access')
print('    to preventive care, not higher inherent clinical risk.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Proxy variable analysis — what the model learns without being told', color=ACCENT, fontsize=12, y=1.02)

# Medicaid rate by race
med_plot = df.groupby('race')['is_medicaid'].mean().sort_values() * 100
bar_colors_m = [DANGER if r not in ['Caucasian', 'Asian'] else MUTED for r in med_plot.index]
axes[0].barh(med_plot.index, med_plot.values, color=bar_colors_m, alpha=0.85)
axes[0].set_xlabel('% on Medicaid')
axes[0].set_title('Medicaid rate by race — payer_code is a race proxy', color=MUTED, fontsize=10)
axes[0].grid(axis='x', alpha=0.3)

# SNF discharge rate by race
snf_plot = df.groupby('race')['discharged_to_snf'].mean().sort_values() * 100
bar_colors_s = [DANGER if r not in ['Caucasian', 'Asian'] else MUTED for r in snf_plot.index]
axes[1].barh(snf_plot.index, snf_plot.values, color=bar_colors_s, alpha=0.85)
axes[1].set_xlabel('% discharged to SNF')
axes[1].set_title('SNF discharge rate by race — discharge_disposition is a race proxy', color=MUTED, fontsize=10)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Train the Biased Model

Features include `race`, `gender`, and `age` directly **and** all four proxy variables.

In [ ]:
# Encode categoricals
cat_cols = [
    'race', 'gender', 'age', 'payer_code', 'medical_specialty',
    'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult',
    'metformin', 'insulin', 'change', 'diabetesMed'
]
df_enc = df.copy()
le = LabelEncoder()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

print('Categorical columns encoded.')

In [ ]:
biased_features = [
    'race',                      # protected attribute ✗
    'gender',                    # protected attribute ✗
    'age',                       # protected attribute ✗
    'payer_code',                # proxy: encodes income + race ✗
    'discharge_disposition_id',  # proxy: encodes insurance/wealth ✗
    'medical_specialty',         # proxy: encodes insurance access ✗
    'number_inpatient',          # proxy: encodes preventive care access ✗
    'admission_type_id',
    'admission_source_id',
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_diagnoses',
    'max_glu_serum',
    'A1Cresult',
    'insulin',
    'change',
    'diabetesMed',
    'diag_1',
    'diag_2',
    'diag_3',
]

X_biased = df_enc[biased_features]
y = df_enc['target']

X_train, X_test, y_train, y_test = train_test_split(
    X_biased, y, test_size=0.2, random_state=42
)

biased_model = RandomForestClassifier(n_estimators=100, random_state=42)
biased_model.fit(X_train, y_train)
biased_preds    = biased_model.predict(X_test)
biased_accuracy = accuracy_score(y_test, biased_preds)

results_b = X_test.copy()
results_b['pred']        = biased_preds
results_b['is_female']   = df.loc[X_test.index, 'is_female'].values
results_b['is_minority'] = df.loc[X_test.index, 'is_minority'].values
results_b['is_elderly']  = df.loc[X_test.index, 'is_elderly'].values

sex_b  = results_b.groupby('is_female')['pred'].mean() * 100
race_b = results_b.groupby('is_minority')['pred'].mean() * 100
age_b  = results_b.groupby('is_elderly')['pred'].mean() * 100

print(f'Model Accuracy: {biased_accuracy:.2%}')
print()
print('— High-Risk Flag Rate by Gender ————————————————————')
print(f'  Male patients      : {sex_b[0]:.2f}% flagged high-risk')
print(f'  Female patients    : {sex_b[1]:.2f}% flagged high-risk')
sex_gap_b = abs(sex_b[0] - sex_b[1])
print(f'\n  Fairness Gap (Gender): {sex_gap_b:.2f}%')
print()
print('— High-Risk Flag Rate by Race ———————————————————————')
print(f'  Caucasian/Asian    : {race_b[0]:.2f}% flagged high-risk')
print(f'  Other minorities   : {race_b[1]:.2f}% flagged high-risk')
race_gap_b = abs(race_b[0] - race_b[1])
print(f'\n  Fairness Gap (Race): {race_gap_b:.2f}%')
print()
print('— High-Risk Flag Rate by Age ————————————————————————')
print(f'  Under 70           : {age_b[0]:.2f}% flagged high-risk')
print(f'  70+ (elderly)      : {age_b[1]:.2f}% flagged high-risk')
age_gap_b = abs(age_b[0] - age_b[1])
print(f'\n  Fairness Gap (Age): {age_gap_b:.2f}%')

## 5. Train the Fair Model

All three protected attributes **and** all four proxy variables are removed. Only objective clinical signals from the current admission remain.

In [ ]:
fair_features = [
    # race                    removed (protected attribute)
    # gender                  removed (protected attribute)
    # age                     removed (protected attribute)
    # payer_code              removed (proxy: encodes income + race)
    # discharge_disposition_id removed (proxy: encodes post-acute access)
    # medical_specialty       removed (proxy: encodes insurance/geography)
    # number_inpatient        removed (proxy: encodes preventive care access gap)
    'admission_type_id',       # retained: emergency vs elective admission
    'admission_source_id',     # retained: ER vs referral vs transfer
    'time_in_hospital',        # retained: length of stay — severity proxy
    'num_lab_procedures',      # retained: diagnostic intensity this visit
    'num_procedures',          # retained: clinical procedures this visit
    'num_medications',         # retained: medication burden this visit
    'number_outpatient',       # retained: outpatient visits
    'number_emergency',        # retained: emergency visits (acute events)
    'number_diagnoses',        # retained: comorbidity count
    'max_glu_serum',           # retained: glucose reading this admission
    'A1Cresult',               # retained: HbA1c — direct diabetes control measure
    'insulin',                 # retained: treatment decision this visit
    'change',                  # retained: medication change flag
    'diabetesMed',             # retained: on diabetes medication flag
    'diag_1',                  # retained: primary ICD diagnosis code
    'diag_2',                  # retained: secondary ICD diagnosis code
    'diag_3',                  # retained: tertiary ICD diagnosis code
]

# Encode only the retained categorical columns for fair model
cat_cols_fair = [
    'diag_1', 'diag_2', 'diag_3',
    'max_glu_serum', 'A1Cresult',
    'insulin', 'change', 'diabetesMed'
]
df_enc_fair = df.copy()
le2 = LabelEncoder()
for col in cat_cols_fair:
    df_enc_fair[col] = le2.fit_transform(df_enc_fair[col].astype(str))

X_fair = df_enc_fair[fair_features]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fair, y, test_size=0.2, random_state=42
)

fair_model = RandomForestClassifier(n_estimators=100, random_state=42)
fair_model.fit(X_train_f, y_train_f)
fair_preds    = fair_model.predict(X_test_f)
fair_accuracy = accuracy_score(y_test_f, fair_preds)

results_f = X_test_f.copy()
results_f['pred']        = fair_preds
results_f['is_female']   = df.loc[X_test_f.index, 'is_female'].values
results_f['is_minority'] = df.loc[X_test_f.index, 'is_minority'].values
results_f['is_elderly']  = df.loc[X_test_f.index, 'is_elderly'].values

sex_f  = results_f.groupby('is_female')['pred'].mean() * 100
race_f = results_f.groupby('is_minority')['pred'].mean() * 100
age_f  = results_f.groupby('is_elderly')['pred'].mean() * 100

print(f'Model Accuracy: {fair_accuracy:.2%}')
print()
print('— High-Risk Flag Rate by Gender ————————————————————')
print(f'  Male patients      : {sex_f[0]:.2f}% flagged high-risk')
print(f'  Female patients    : {sex_f[1]:.2f}% flagged high-risk')
sex_gap_f = abs(sex_f[0] - sex_f[1])
print(f'\n  New Fairness Gap (Gender): {sex_gap_f:.2f}%')
print()
print('— High-Risk Flag Rate by Race ———————————————————————')
print(f'  Caucasian/Asian    : {race_f[0]:.2f}% flagged high-risk')
print(f'  Other minorities   : {race_f[1]:.2f}% flagged high-risk')
race_gap_f = abs(race_f[0] - race_f[1])
print(f'\n  New Fairness Gap (Race): {race_gap_f:.2f}%')
print()
print('— High-Risk Flag Rate by Age ————————————————————————')
print(f'  Under 70           : {age_f[0]:.2f}% flagged high-risk')
print(f'  70+ (elderly)      : {age_f[1]:.2f}% flagged high-risk')
age_gap_f = abs(age_f[0] - age_f[1])
print(f'\n  New Fairness Gap (Age): {age_gap_f:.2f}%')

## 6. Compare Results

In [ ]:
# Recompute gaps using the actual scalar values from each model's results
gap_sex_b  = abs(sex_b[0]  - sex_b[1])
gap_sex_f  = abs(sex_f[0]  - sex_f[1])
gap_race_b = abs(race_b[0] - race_b[1])
gap_race_f = abs(race_f[0] - race_f[1])
gap_age_b  = abs(age_b[0]  - age_b[1])
gap_age_f  = abs(age_f[0]  - age_f[1])

# Gender gap increased — show direction correctly
def gap_change(before, after):
    if before == 0:
        return 'N/A'
    pct = (after - before) / before * 100
    if pct < 0:
        return f'{abs(pct):.1f}% reduction'
    else:
        return f'+{pct:.1f}% increase'

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Healthcare Readmission — Biased vs Mitigated Model', color=ACCENT, fontsize=13, y=1.02)

dims = [
    (axes[0], [sex_b[0],  sex_b[1]],  [sex_f[0],  sex_f[1]],
     ['Male', 'Female'],               gap_sex_b,  gap_sex_f,  'Gender'),
    (axes[1], [race_b[0], race_b[1]], [race_f[0], race_f[1]],
     ['Caucasian/Asian', 'Minorities'], gap_race_b, gap_race_f, 'Race'),
    (axes[2], [age_b[0],  age_b[1]],  [age_f[0],  age_f[1]],
     ['Under 70', '70+ elderly'],      gap_age_b,  gap_age_f,  'Age'),
]

for ax, vals_b, vals_f, groups, gb, gf, label in dims:
    x = np.arange(len(groups))
    width = 0.35
    bars_b = ax.bar(x - width/2, vals_b, width, color=DANGER, label='Biased',    alpha=0.85)
    bars_f = ax.bar(x + width/2, vals_f, width, color=SAFE,   label='Mitigated', alpha=0.85)
    for bar, val in list(zip(bars_b, vals_b)) + list(zip(bars_f, vals_f)):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
                f'{val:.2f}%', ha='center', fontsize=9, color=ACCENT)
    ax.set_xticks(x)
    ax.set_xticklabels(groups, fontsize=9)
    ax.set_ylim(0, max(max(vals_b), max(vals_f)) * 1.4)
    ax.set_ylabel('% flagged high-risk')
    ax.set_title(f'{label}: {gb:.2f}% → {gf:.2f}%', color=MUTED, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nSummary')
print(f'-------')
print(f'Gender gap  before: {gap_sex_b:.2f}%   after: {gap_sex_f:.2f}%   {gap_change(gap_sex_b, gap_sex_f)}')
print(f'Race gap    before: {gap_race_b:.2f}%   after: {gap_race_f:.2f}%   {gap_change(gap_race_b, gap_race_f)}')
print(f'Age gap     before: {gap_age_b:.2f}%   after: {gap_age_f:.2f}%   {gap_change(gap_age_b, gap_age_f)}')
print()
print('Note: gender gap increased slightly after proxy removal.')
print('Proxy variables did not carry strong gender signal — their')
print('removal shifted prediction patterns in a way that widened')
print('the gender gap by ~0.02pp. Race and age gaps reduced substantially.')

## Key Insight

Removing `race`, `gender`, and `age` alone is not enough. Four proxy variables reconstruct protected-class signal even after the demographic columns are gone.

`payer_code` is the most structurally loaded: Medicaid status encodes poverty, and poverty is racially stratified by decades of structural inequality in employment and insurance access. `discharge_disposition_id` encodes whether a patient can access a skilled nursing facility — a function of insurance coverage and geographic proximity, not clinical trajectory. Where a patient goes after discharge depends on what their insurer will cover and whether there is a facility nearby: a model that learns "this patient goes home → higher readmission risk" is encoding a resource gap as a patient-level risk factor. `medical_specialty` access is determined upstream by insurance and geography. And `number_inpatient` carries racial signal because AfricanAmerican patients average 0.70 prior hospitalisations vs 0.48 for Asian patients — a gap that reflects underinvestment in preventive care in minority communities, not higher individual clinical risk.

One result is honest and worth noting: the gender gap increased slightly (from 0.02% to 0.04%) after proxy removal. The proxy variables removed did not carry significant gender signal — their removal shifted the model's prediction landscape in a way that slightly widened the male/female gap. This is a real outcome of this specific mitigation, not an artefact. Race and age gaps reduced substantially: 25% and 68% respectively.

**The fix:** Drop everything that encodes structural inequality as individual risk. Retain only the clinical record of this admission — diagnosis codes, lab and procedure counts, glucose control, medication management, and how the patient arrived. These are the signals a clinician would use to assess readmission risk without reference to who the patient is demographically.

---

*Part of the [Fair Code project](https://github.com/yakew7/Fair-Code) by [@thefaircodeproject](https://instagram.com/thefaircodeproject)*